# Complexity-04b — La semaine de l'algorithmique en ligne : secrétaire matroïdal et k-server

[Source Python](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb) · [Index](README.md) · [04 — Décider sans connaître la suite](Complexity-04-OnlineAlgorithms-Python.ipynb) · [04c — Work function](Complexity-04c-KServer-WorkFunction-Python.ipynb) · [04d — Secrétaire matroïdal](Complexity-04d-Secretaire-Matroidal-Python.ipynb)

**Positionnement.** Ce carnet approfondit 04 : décision irréversible, rapport compétitif, location ou achat, pagination et secrétaire classique. Il ajoute les oracles de matroïde, la fonction de travail sur métriques finies et deux annonces de septembre 2026 que le source Python met en regard. La baseline du secrétaire est reprise ici car les stratégies matroïdales lui répondent ; les sections 2 à 4 en sont une extension de niveau recherche, pas un prérequis du parcours initial.

Le source Python relie deux prépublications : celle attribuée à Sahil Singla sur le secrétaire matroïdal et celle attribuée à Christian Coester, Elias Koutsoupias et Marek Zbysiński sur le k-server. Il les présente comme des annonces rapprochées. Ce port a consulté le **Python versionné** ; il ne revendique pas avoir lu les PDF mentionnés dans ce Python. Les algorithmes et résultats numériques ci-dessous sont exécutés ici, les énoncés généraux sont **rapportés par le source**, et les preuves sont hors du carnet.

**Objectifs.** Formuler la règle du secrétaire et son hypothèse d'ordre aléatoire ; construire deux oracles d'indépendance et comparer des stratégies à l'optimum ; calculer la fonction de travail et opposer WFA au plus proche ; distinguer une mesure d'une garantie pire cas. **Prérequis** : permutations, ensembles et coûts ; pas de bibliothèque d'algorithmique en ligne. **Durée pédagogique** : 45 minutes.

Les implémentations sont natives C# et emploient `System.Random`. Les graines fixent les essais .NET mais ne reproduisent pas les tirages NumPy de la source ; les tailles, données structurelles et budgets restent ceux du source.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;
using System.Globalization;

// Une culture invariante rend les nombres lisibles de la même façon dans chaque soumission.
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;
Console.WriteLine("Structures et générateur .NET prêts");


Structures et générateur .NET prêts


## 1. Le secrétaire classique — la baseline à battre

$n$ candidats arrivent en ordre aléatoire. On ne connaît que leurs rangs relatifs parmi les personnes déjà rencontrées ; accepter est irréversible, et le but est d'accepter le meilleur. La règle classique observe sans accepter les $\lfloor n/e\rfloor$ premiers candidats, puis accepte le premier qui bat le maximum de l'échantillon. Le source rapporte que sa probabilité de succès tend vers $1/e\approx0{,}368$ lorsque $n$ grandit.

Cette limite est asymptotique : à $n=20$, une probabilité plus haute n'est pas une contradiction. La cellule tire 20 000 permutations pour chacune des tailles 20, 100 et 1000. Elle ne nécessite que les rangs : changer la distribution continue de valeurs sans changer l'ordre relatif ne change pas la règle. C'est la référence à laquelle le secrétaire **matroïdal**, où plusieurs choix sont possibles mais contraints, sera comparé.

In [2]:
var rng = new Random(42);
// Fisher–Yates génère un ordre d’arrivée uniforme sans modifier les poids.
int[] Permutation(int n, Random r)
{
    var p = Enumerable.Range(0, n).ToArray();
    for (int i = n - 1; i > 0; i--)
    {
        int j = r.Next(i + 1);
        (p[i], p[j]) = (p[j], p[i]);
    }

    return p;
}

// Après observation, la première valeur supérieure au maximum vu est définitive.
bool Secretaire(int[] p)
{
    int seuil = Math.Max(1, (int)(p.Length / Math.E));
    int meilleur = p.Take(seuil).Max();
    foreach (int x in p.Skip(seuil))
        if (x > meilleur)
            return x == p.Length - 1;
    return false;
}

// Chaque essai utilise une nouvelle permutation et compte seulement le meilleur candidat.
double MonteCarlo(int n, int essais, Random r)
{
    int succes = 0;
    for (int i = 0; i < essais; i++)
        if (Secretaire(Permutation(n, r)))
            succes++;
    return (double)succes / essais;
}

foreach (int n in new[]
{
    20,
    100,
    1000
}

)
    Console.WriteLine($"n={n} : {MonteCarlo(n, 20000, rng):F4} (limite 1/e={1 / Math.E:F4})");


n=20 : 0.3833 (limite 1/e=0.3679)


n=100 : 0.3771 (limite 1/e=0.3679)


n=1000 : 0.3690 (limite 1/e=0.3679)


### Lecture — une limite, trois tailles finies

La sortie imprime une estimation de la fréquence du meilleur pour chaque taille et le repère $1/e$. L'estimation dépend des permutations effectivement tirées, et son bruit est fini : des écarts ne démontrent ni une défaillance de la règle ni une amélioration universelle. Le rapprochement des trois tailles avec la limite prépare la section suivante : si accepter trop tôt occupe une place rare, attendre trop longtemps peut aussi faire manquer des éléments utiles.

Les fréquences de cette exécution sont **0,3833** pour $n=20$, **0,3771** pour $n=100$ et **0,3690** pour $n=1000$, contre le repère **0,3679**. Le rapprochement observé sur ces trois valeurs reste une estimation : les petites tailles ont une correction finie et toutes les fréquences ont un bruit de Monte Carlo.

## 2. Le secrétaire matroïdal — choisir sous contrainte

Les éléments arrivent en ordre aléatoire, chacun porte une valeur positive ; il faut décider immédiatement et irréversiblement. Les éléments acceptés doivent rester **indépendants** selon un matroïde connu par un oracle. L'objectif est de maximiser l'espérance de la somme acceptée, comparée à l'optimum hors ligne qui voit toutes les valeurs. Sur un matroïde à poids positifs, le glouton hors ligne qui parcourt les valeurs décroissantes donne cet optimum.

Le source rapporte la conjecture de Babaioff, Immorlica et Kleinberg : obtenir une constante $\alpha>0$ indépendante de la taille et du matroïde telle que $\mathbb E[ALG]\geq\alpha\,OPT$. Il rapporte aussi l'annonce attribuée à Singla : une probabilité d'acceptation au moins $1/4$ pour **chaque** élément d'un optimum, avec pour interface le nombre d'éléments et l'oracle d'indépendance. Nous n'exécutons ni la preuve, ni l'algorithme de Singla dans ce carnet synthétique : [04d Python](Complexity-04d-Secretaire-Matroidal-Python.ipynb) en donne une étude séparée.

Deux exemples d'oracles rendent la contrainte concrète : un matroïde **uniforme** autorise au plus trois éléments parmi douze ; un matroïde **graphique** accepte des arêtes tant qu'elles ne ferment pas un cycle dans une grille $3\times4$. Sur chacun, nous confrontons le glouton en ligne qui accepte tout élément admissible et une stratégie qui observe la moitié des arrivées puis accepte les éléments admissibles de valeur au moins la médiane observée. Les deux sont des références, sans garantie générale affirmée ici.

In [3]:
// L’oracle uniforme ne dépend que du cardinal ; l’oracle graphique interdit les cycles.
bool IndepUniforme(IEnumerable<int> indices, int rang) => indices.Count() <= rang;
bool IndepGraphique(IEnumerable<int> indices, (int, int)[] aretes)
{
    var parent = new Dictionary<int, int>();
    int Racine(int x)
    {
        if (!parent.ContainsKey(x))
            parent[x] = x;
        while (parent[x] != x)
            x = parent[x];
        return x;
    }

    foreach (int i in indices)
    {
        var(a, b) = aretes[i];
        int ra = Racine(a), rb = Racine(b);
        if (ra == rb)
            return false;
        parent[ra] = rb;
    }

    return true;
}

(double, List<int>) Optimum(double[] valeurs, Func<IEnumerable<int>, bool> ind)
{
    var choix = new List<int>();
    foreach (int i in Enumerable.Range(0, valeurs.Length).OrderByDescending(i => valeurs[i]))
    {
        var candidat = choix.Append(i);
        if (ind(candidat))
            choix.Add(i);
    }

    return (choix.Sum(i => valeurs[i]), choix);
}

// La médiane vient exclusivement de la phase d’observation, sans décision rétrospective.
double Strategie(double[] valeurs, Func<IEnumerable<int>, bool> ind, bool seuil, Random r)
{
    var ordre = Permutation(valeurs.Length, r);
    int debut = seuil ? valeurs.Length / 2 : 0;
    double mediane = 0;
    if (seuil)
    {
        var echantillon = ordre.Take(debut).Select(i => valeurs[i]).OrderBy(x => x).ToArray();
        mediane = (echantillon[(debut - 1) / 2] + echantillon[debut / 2]) / 2;
    }

    var choix = new List<int>();
    foreach (int i in ordre.Skip(debut))
        if ((!seuil || valeurs[i] >= mediane) && ind(choix.Append(i)))
            choix.Add(i);
    return choix.Sum(i => valeurs[i]);
}

// Comparer la valeur moyenne obtenue à l’optimum fixe de cette instance.
double Competitivite(double[] valeurs, Func<IEnumerable<int>, bool> ind, bool seuil, int essais, Random r)
{
    double total = 0, opt = Optimum(valeurs, ind).Item1;
    for (int i = 0; i < essais; i++)
        total += Strategie(valeurs, ind, seuil, r);
    return total / essais / opt;
}

var valU = Enumerable.Range(0, 12).Select(_ => 1 + 9 * rng.NextDouble()).ToArray();
var ar = new List<(int, int)>();
for (int l = 0; l < 3; l++)
    for (int c = 0; c < 4; c++)
    {
        int v = l * 4 + c;
        if (c < 3)
            ar.Add((v, v + 1));
        if (l < 2)
            ar.Add((v, v + 4));
    }

var aretes = ar.ToArray();
var valG = aretes.Select(_ => 1 + 9 * rng.NextDouble()).ToArray();
Func<IEnumerable<int>, bool> uniforme = s => IndepUniforme(s, 3);
Func<IEnumerable<int>, bool> graphique = s => IndepGraphique(s, aretes);
foreach (var inst in new[]
{
    ("uniforme", valU, uniforme),
    ("graphique", valG, graphique)
}

)
{
    Console.WriteLine($"{inst.Item1} OPT={Optimum(inst.Item2, inst.Item3).Item1:F2} ; pur={Competitivite(inst.Item2, inst.Item3, false, 2000, rng):F4} ; seuil={Competitivite(inst.Item2, inst.Item3, true, 2000, rng):F4}");
}

Console.WriteLine($"Grille 3×4 : {aretes.Length} arêtes ; 2000 essais par stratégie");


uniforme OPT=25.95 ; pur=0.6948 ; seuil=0.7265


graphique OPT=79.05 ; pur=0.7664 ; seuil=0.4391


Grille 3×4 : 17 arêtes ; 2000 essais par stratégie


### Lecture — la structure change le dilemme

La cellule précédente imprime l'optimum et les ratios de gain moyen des deux stratégies, 2 000 essais par stratégie. La grille possède 17 arêtes. Sur le tirage .NET observé, la stratégie à seuil favorise la sélection de poids élevés dans l'uniforme ; la phase d'attente écarte au contraire de nombreuses arêtes utiles pour le graphique, où le glouton pur obtient ici davantage. Il faut lire ces constats **sur ces poids et ces deux instances**, non comme une preuve qu'une stratégie domine sur une famille entière.

La section suivante rejoue le cas uniforme sur huit graines indépendantes tout en conservant les mêmes poids. La dispersion sépare la variabilité de l'ordre d'arrivée de celle de l'instance elle-même. L'algorithme annoncé par le source pour une garantie uniforme est d'une autre nature : un banc de deux heuristiques mesure le dilemme entre accepter tôt et préserver de la capacité, il ne démontre pas cette garantie.

Sur les poids tirés ici, l'optimum uniforme vaut **25,95** : les ratios estimés sont **0,6948** pour le pur et **0,7265** pour le seuil. Sur la grille, l'optimum vaut **79,05** et les ratios sont respectivement **0,7664** et **0,4391**. La direction s'inverse entre ces deux structures ; les nombres ne sont pas ceux d'un autre générateur ni une comparaison de performances Python/.NET.

In [4]:
// Les huit graines changent les permutations, pas les poids de l’instance uniforme.
var graines = new[]
{
    0,
    1,
    7,
    42,
    99,
    123,
    256,
    1024
};
var purs = new List<double>();
var seuils = new List<double>();
foreach (int seed in graines)
{
    var r = new Random(seed);
    purs.Add(Competitivite(valU, uniforme, false, 2000, r));
    seuils.Add(Competitivite(valU, uniforme, true, 2000, r));
}

// Écart-type de population : dispersion des huit moyennes empiriques.
double EcartType(List<double> x)
{
    double m = x.Average();
    return Math.Sqrt(x.Average(v => (v - m) * (v - m)));
}

Console.WriteLine($"8 graines : pur [{purs.Min():F4}, {purs.Max():F4}], moyenne {purs.Average():F4}, écart-type {EcartType(purs):F4}");
Console.WriteLine($"8 graines : seuil [{seuils.Min():F4}, {seuils.Max():F4}], moyenne {seuils.Average():F4}, écart-type {EcartType(seuils):F4}; seuil devant {Enumerable.Range(0, 8).Count(i => seuils[i] > purs[i])}/8");


8 graines : pur [0.6946, 0.7061], moyenne 0.7001, écart-type 0.0034


8 graines : seuil [0.7123, 0.7427], moyenne 0.7274, écart-type 0.0080; seuil devant 8/8


### Lecture — huit graines, une seule instance

La sortie donne l'intervalle, la moyenne, l'écart-type et le nombre de graines où le seuil est devant. Ces valeurs sont recalculées sur les poids du tirage .NET et ne sont pas les valeurs NumPy publiées dans le source. La répétition mesure la dispersion de l'ordre pour une instance **fixe** ; elle n'échantillonne pas l'infinité des matroïdes ni les lois de poids. En particulier, « aucune des deux ne domine ces deux instances » est une observation, et non un théorème sur tous les algorithmes à seuil.

Le pur varie ici de **0,6946 à 0,7061**, moyenne **0,7001** et écart-type **0,0034** ; le seuil varie de **0,7123 à 0,7427**, moyenne **0,7274** et écart-type **0,0080**. Sur cette instance, le seuil est devant pour **8/8** graines. L'écart-type décrit ces huit moyennes empiriques, pas une estimation de l'incertitude sur tous les matroïdes.

## Exercice 1 — Oracle de partition

Définir l'indépendance par quotas de couleur, puis évaluer la stratégie à seuil. Tester notamment l'ensemble vide, deux éléments de même couleur et deux couleurs distinctes.

**Contexte.** Une couleur désigne une classe d'éléments, et chaque classe possède son propre quota. Ce troisième matroïde prolonge les oracles uniforme et graphique : l'uniforme impose une capacité globale, la partition plusieurs capacités locales, et le graphique l'acyclicité.

**Méthode.** Compter les indices de la sélection par couleur, comparer chaque effectif au quota associé et retourner le verdict. L'ensemble vide est indépendant ; avec quatre éléments de couleurs 0,0,1,1 et deux quotas égaux à 1, les indices 0 et 1 sont interdits ensemble tandis que 0 et 2 sont permis. Une fois l'oracle complété, la cellule calcule l'optimum et le ratio de la stratégie à seuil sur les douze valeurs du banc, avec 2 000 ordres aléatoires. Expliquer si la contrainte locale change le compromis observé.

In [5]:
bool? IndepPartition(int[] indices, int[] couleurs, int[] quotas)
{
    // TODO : compter les éléments par couleur et comparer aux quotas.
    return null;
}

var ex1a = IndepPartition(new[] { 0, 1 }, new[] { 0, 0, 1, 1 }, new[] { 1, 1 });
if (ex1a == null)
    Console.WriteLine("Exercice 1 à compléter");
else
{
    if (ex1a != false || IndepPartition(new[] { 0, 2 }, new[] { 0, 0, 1, 1 }, new[] { 1, 1 }) != true || IndepPartition(Array.Empty<int>(), new[] { 0, 0, 1, 1 }, new[] { 1, 1 }) != true)
        throw new Exception("Oracle de partition incorrect");
    Console.WriteLine("Oracle de partition : cas vide, conflit et deux couleurs vérifiés");
}

if (ex1a != null)
{
    int[] couleurs =
    {
        0,
        0,
        0,
        1,
        1,
        1,
        2,
        2,
        2,
        2,
        2,
        2
    };
    int[] quotas =
    {
        1,
        2,
        1
    };
    Func<IEnumerable<int>, bool> oraclePartition = s => IndepPartition(s.ToArray(), couleurs, quotas) == true;
    double ratio = Competitivite(valU, oraclePartition, true, 2000, new Random(71));
    Console.WriteLine($"Partition : OPT={Optimum(valU, oraclePartition).Item1:F3}, seuil/OPT={ratio:F3} (2000 essais)");
}


Exercice 1 à compléter


## 3. k-server — une fonction de travail exacte

$k$ serveurs occupent des points d'un espace métrique. À chaque requête, un serveur doit se trouver au point demandé ; s'il faut le déplacer, on paie la distance. L'algorithme en ligne ignore les demandes futures, tandis que l'optimum hors ligne connaît la séquence entière. La comparaison peut inclure une constante additive : $ALG\le c\,OPT+\beta$, où $\beta$ ne dépend pas de la séquence. Un ratio fini $ALG/OPT$ sur une instance n'établit pas une garantie compétitive générale.

Le source rapporte une borne inférieure $k$ pour les algorithmes déterministes dans le cadre décrit, l'ancienne garantie $(2k-1)$ du work function algorithm (WFA), puis l'annonce de prépublication selon laquelle WFA atteindrait $k$ sur toute métrique. Ces résultats généraux sont **rapportés par le Python** et ne sont pas démontrés ici.

Pour $k=2$, nous énumérons les $\binom{m}{2}$ configurations, deux serveurs à des points distincts. La distance entre deux configurations est le meilleur des deux appariements des serveurs. La fonction de travail $W_t(S)$ est le coût minimal pour servir les $t$ premières requêtes et finir en $S$ : on initialise par la distance à la configuration initiale, puis on minimise $W_{t-1}(S')+d(S',S)$ sur les configurations précédentes, en imposant que $S$ couvre la requête courante. $\min_S W_T(S)$ donne l'optimum hors ligne.

WFA emploie la **même mémoire du passé** pour choisir à l'instant $t$ : parmi les mouvements admissibles qui servent la requête, il minimise le mouvement immédiat plus la fonction de travail résultante. Le calcul ne regarde pas les demandes suivantes. La ligne utilise $d(i,j)=|i-j|$ ; le cycle $C_5$ prend le plus court des deux sens.

In [6]:
List<(int, int)> Configs(int m)
{
    var c = new List<(int, int)>();
    for (int a = 0; a < m; a++)
        for (int b = a + 1; b < m; b++)
            c.Add((a, b));
    return c;
}

double DistanceConfig((int, int) a, (int, int) b, Func<int, int, int> d) => Math.Min(d(a.Item1, b.Item1) + d(a.Item2, b.Item2), d(a.Item1, b.Item2) + d(a.Item2, b.Item1));
(double wfa, double opt) JouerWfa(int m, (int, int) init, int[] req, Func<int, int, int> d)
{
    var configs = Configs(m);
    var w = configs.ToDictionary(c => c, c => DistanceConfig(init, c, d));
    var courant = init;
    double cout = 0;
    foreach (int r in req)
    {
        var suivant = new Dictionary<(int, int), double>();
        foreach (var a in configs.Where(c => c.Item1 == r || c.Item2 == r))
            suivant[a] = w.Min(kv => kv.Value + DistanceConfig(kv.Key, a, d));
        // Le choix courant ne dépend que de la table du préfixe.
        var candidates = new List<(int, int)>();
        if (courant.Item1 == r || courant.Item2 == r)
            candidates.Add(courant);
        else
        {
            candidates.Add((Math.Min(r, courant.Item2), Math.Max(r, courant.Item2)));
            candidates.Add((Math.Min(r, courant.Item1), Math.Max(r, courant.Item1)));
        }

        var choix = candidates.OrderBy(a => DistanceConfig(courant, a, d) + suivant[a]).First();
        cout += DistanceConfig(courant, choix, d);
        courant = choix;
        w = suivant;
    }

    return (cout, w.Values.Min());
}

double JouerGlouton((int, int) init, int[] req, Func<int, int, int> d)
{
    int[] p =
    {
        init.Item1,
        init.Item2
    };
    double cout = 0;
    foreach (int r in req)
    {
        int i = d(p[0], r) <= d(p[1], r) ? 0 : 1;
        cout += d(p[i], r);
        p[i] = r;
    }

    return cout;
}

Func<int, int, int> cycle = (i, j) => Math.Min(Math.Abs(i - j), 5 - Math.Abs(i - j));
Func<int, int, int> ligne = (i, j) => Math.Abs(i - j);
foreach (int repetitions in new[]
{
    30,
    60,
    240
}

)
{
    var req = Enumerable.Range(0, 2 * repetitions).Select(i => i % 2 == 0 ? 3 : 4).ToArray();
    var x = JouerWfa(5, (0, 1), req, cycle);
    double g = JouerGlouton((0, 1), req, cycle);
    Console.WriteLine($"C5 T={req.Length} : OPT={x.opt}, WFA={x.wfa} ratio={x.wfa / x.opt:F3}, glouton={g} ratio={g / x.opt:F1}");
}

var alea = Enumerable.Range(0, 60).Select(_ => rng.Next(8)).ToArray();
var l8 = JouerWfa(8, (0, 7), alea, ligne);
Console.WriteLine($"Ligne 8, T=60 : OPT={l8.opt}, WFA/OPT={l8.wfa / l8.opt:F3}, glouton/OPT={JouerGlouton((0, 7), alea, ligne) / l8.opt:F3}");


C5 T=60 : OPT=3, WFA=5 ratio=1.667, glouton=61 ratio=20.3


C5 T=120 : OPT=3, WFA=5 ratio=1.667, glouton=121 ratio=40.3


C5 T=480 : OPT=3, WFA=5 ratio=1.667, glouton=481 ratio=160.3


Ligne 8, T=60 : OPT=79, WFA/OPT=1.101, glouton/OPT=1.101


### Lecture — un piège pour le plus proche

Sur $C_5$, les serveurs partent de $(0,1)$ et les requêtes alternent $3,4$. L'optimum peut payer une réorganisation puis couvrir les deux points ; le plus proche bascule au contraire de l'un à l'autre. La sortie rapporte les coûts réels, ainsi que les ratios pour 60, 120 et 480 requêtes. Pour la séquence observée, WFA paie 5 lorsque l'optimum paie 3, alors que le coût du plus proche croît avec le nombre d'alternances : la sortie permet de constater un adversaire de cette heuristique.

Sur la ligne à huit points, la séquence de 60 demandes est tirée au hasard ; ses coûts et ratios sont ceux de **ce tirage .NET**. Un régime moyen facile ne remplace pas l'analyse adversariale. À l'inverse, voir WFA sous un repère sur ces quelques séquences ne prouve pas le résultat général attribué par le source : l'analyse théorique et sa constante additive concernent toutes les séquences admissibles.

Plus précisément, le ratio WFA est **5/3 ≈ 1,667** à chacune des trois longueurs ; celui du plus proche est **20,3**, **40,3** puis **160,3** pour 60, 120 et 480 requêtes. La ligne aléatoire de cette exécution donne un optimum **79** et un ratio **1,101** pour chacune des deux stratégies. Ce dernier chiffre dépend de la séquence .NET seedée : il illustre un régime, pas une égalité générale entre algorithmes.

## Exercice 2 — Étoile et stratégie gloutonne

Construire la distance de l'étoile à m feuilles puis chercher une séquence qui coûte cher au plus proche. Appeler WFA avec m+1 points ; comparer les coûts et l'OPT.

Renvoyer aussi une séquence de 60 requêtes sur une étoile à quatre feuilles, avec serveurs initiaux en (1,2), et comparer effectivement les deux coûts à OPT.

**Convention.** L'étoile à $m$ feuilles contient $m+1$ points : 0 est le centre et les feuilles portent les indices 1 à $m$. La distance vaut 0 sur la diagonale, 1 entre centre et feuille, 2 entre deux feuilles distinctes. Il faut passer $m+1$ comme nombre de points à `JouerWfa`, contrairement à la ligne et au cycle où le paramètre désigne directement ce nombre.

**Recherche.** Avec deux serveurs sur deux feuilles, chercher une alternance entre deux autres feuilles qui force le plus proche à se déplacer souvent. Le contrôle effectue 60 requêtes, calcule les coûts payés et l'optimum par la même DP exacte ; la séquence doit rendre le plus proche sensiblement plus coûteux que l'optimum. Expliquer le mécanisme, puis confronter WFA au repère $k=2$ sur cette instance. Cette vérification n'est pas une preuve de la garantie pour toutes les étoiles.

In [7]:
int? DistanceEtoile(int a, int b, int m)
{
    // TODO : diagonale nulle, centre-feuille 1, deux feuilles distinctes 2.
    return null;
}

int[] SuiteEtoile()
{
    // TODO : construire une séquence adversariale de 60 requêtes sur les feuilles 1..4.
    return null;
}

var ex2 = DistanceEtoile(0, 2, 3);
if (ex2 == null)
    Console.WriteLine("Exercice 2 à compléter");
else
{
    if (ex2 != 1 || DistanceEtoile(1, 2, 3) != 2 || DistanceEtoile(2, 2, 3) != 0)
        throw new Exception("Distance de l'étoile incorrecte");
    Console.WriteLine("Étoile : centre, feuilles distinctes et diagonale vérifiés");
}

if (ex2 != null)
{
    var suite = SuiteEtoile();
    if (suite == null)
        throw new Exception("Séquence adversariale manquante après définition de la distance");
    else
    {
        if (suite.Length != 60 || suite.Any(x => x < 1 || x > 4))
            throw new Exception("Séquence de 60 feuilles attendue");
        Func<int, int, int> etoile = (a, b) => DistanceEtoile(a, b, 4).Value;
        var w = JouerWfa(5, (1, 2), suite, etoile);
        double g = JouerGlouton((1, 2), suite, etoile);
        if (w.opt <= 0 || g / w.opt < 5 || w.wfa > 2 * w.opt + 1e-9)
            throw new Exception("Séquence non discriminante ou WFA incohérent");
        Console.WriteLine($"Étoile T=60 : OPT={w.opt}, WFA={w.wfa}, glouton={g}, ratio glouton={g / w.opt:F2}");
    }
}


Exercice 2 à compléter


## 4. Deux modèles d'incertitude

| | Secrétaire matroïdal | k-server |
|---|---|---|
| Entrée | Arrivée aléatoire | Séquence adversariale |
| Décision | Accepter ou refuser irréversiblement | Déplacer un serveur et payer une distance |
| Référence | Espérance de la valeur, optimum hors ligne | Coût total, optimum hors ligne |
| Stratégies du banc | Glouton pur ou seuil, **pas** l'algorithme de Singla | WFA exact sur petites métriques, face au plus proche |
| Résultat rapporté | Chaque élément de l'optimum serait accepté avec probabilité au moins 1/4 | WFA serait k-compétitif sur toute métrique |

L'analogie rassemble des problèmes d'information incomplète, mais leurs hypothèses diffèrent. Le secrétaire suppose un ordre aléatoire et évalue une espérance ; k-server accepte une suite adversariale et compare des coûts cumulés. Le premier exploite la randomisation ; WFA est un algorithme déterministe dont l'analyse concerne le pire cas. Les deux conjectures rapportées par le source sont datées respectivement de 2007 et d'environ 1990.

Ces deux dernières lignes rapportent les annonces citées dans le notebook Python, pas des preuves exécutées ici. Pour k-server, une garantie compétitive peut s'écrire coût(ALG) ≤ c·OPT + β, avec β indépendant de la suite. Le banc ne teste que des coûts et ratios finis. Le récapitulatif rejoue les essais supplémentaires du source.

Le source décrit aussi un état historique de Mathlib v4.33.0 : une théorie des matroïdes est présente, tandis que les couches d'algorithmes en ligne, k-server et ratios compétitifs ne sont pas identifiées dans son relevé. Ce constat est rapporté au moment de rédaction du source, sans nouvel audit de Mathlib ici.

### Prudence épistémique

Le source distingue trois niveaux. **Mesuré** : les fréquences du secrétaire, les comparaisons des deux oracles et les coûts WFA/glouton produits par ce carnet. **Rapporté** : les énoncés attribués aux deux prépublications, présentés par le source comme annonces de leurs auteurs. **Absent** : les preuves, leur vérification ligne à ligne et toute formalisation mécanique ici. Les textes du source étaient écrits quelques jours après les dépôts ; leur prudence sur l'évaluation par les pairs concerne ce contexte historique. Même si une annonce changeait de statut, les algorithmes et expériences resteraient rejouables.

### Mathlib — relevé historique du source

Le Python compare son contenu avec un relevé de Mathlib v4.33.0 effectué lors de sa rédaction. Il rapporte des définitions de matroïdes, indépendance et rang dans `Mathlib.Combinatorics.Matroid`, mais ne relève pas de couche complète pour algorithmes en ligne, ratio compétitif, secrétaire ou k-server. La fonction de travail sur un espace fini est calculable comme notre DP ; la preuve du ratio général demanderait des objets et arguments supplémentaires. Il s'agit d'un relevé daté **rapporté par le Python**, pas d'une nouvelle recherche exhaustive menée pour ce port.

In [8]:
Console.WriteLine($"Secrétaire n=1000, 5000 essais : {MonteCarlo(1000, 5000, rng):F4}");
Console.WriteLine($"Uniforme : pur {Competitivite(valU, uniforme, false, 2000, rng):F4}, seuil {Competitivite(valU, uniforme, true, 2000, rng):F4}");
Console.WriteLine("C5 : voir les trois longueurs et les coûts réels ci-dessus");


Secrétaire n=1000, 5000 essais : 0.3738


Uniforme : pur 0.6982, seuil 0.7283


C5 : voir les trois longueurs et les coûts réels ci-dessus


## Exercice 3 — Valeurs log-normales

Écrire un générateur de n valeurs log-normales indépendantes de paramètres mu et sigma, puis appliquer la règle du secrétaire aux valeurs obtenues. Estimer le succès sur 3 000 essais. Les contrôles distinguent explicitement la loi des valeurs et la règle de sélection.

**Indice sur la génération.** Si $Z$ est normale standard, $\exp(\mu+\sigma Z)$ est log-normale. Deux tirages uniformes indépendants permettent de construire $Z$ par Box–Muller. Le contrôle examine d'abord des listes déterministes pour vérifier la règle du premier record, puis la moyenne et la variance des logarithmes générés ; il termine par 3 000 expériences de taille 100.

**Question.** Pour des valeurs indépendantes d'une loi continue, les rangs forment une permutation uniforme. La queue lourde change les valeurs absolues et leur moyenne, mais quelle information de ces valeurs la règle du secrétaire utilise-t-elle ? Expliquer pourquoi le succès attendu se compare encore à la baseline de la section 1, à l'erreur d'échantillonnage près.

In [9]:
double[] ValeursLogNormales(int n, double mu, double sigma, Random r)
{
    // TODO : générer des normales indépendantes, par exemple avec Box–Muller, puis exponentier.
    return null;
}

bool? SecretaireValeurs(double[] valeurs)
{
    // TODO : ignorer les floor(n/e) premières valeurs puis sélectionner le premier record.
    return null;
}

var exemple = ValeursLogNormales(8, 0, 1, new Random(4));
var exempleDecision = SecretaireValeurs(new[] { 1.0, 2, 4, 3, 5, 6 });
if (exemple == null && exempleDecision == null)
    Console.WriteLine("Exercice 3 à compléter");
else if (exemple == null || exempleDecision == null)
    throw new Exception("Les deux fonctions doivent être complétées");
else
{
    if (SecretaireValeurs(new[] { 1.0, 2, 4, 3, 5, 6 }) != false || SecretaireValeurs(new[] { 1.0, 2, 6, 3, 5, 4 }) != true)
        throw new Exception("Règle du premier record incorrecte");
    for (int j = 0; j < 20; j++)
        if (SecretaireValeurs(new[] { 1.0, 2, 4, 3, 5, 6 }) != false || SecretaireValeurs(new[] { 1.0, 2, 6, 3, 5, 4 }) != true)
            throw new Exception("La décision doit suivre les valeurs, sans tirage indépendant");
    var loi = ValeursLogNormales(12000, 0.7, 0.8, new Random(99));
    if (loi == null || loi.Length != 12000 || loi.Any(x => x <= 0 || !double.IsFinite(x)))
        throw new Exception("Valeurs log-normales invalides");
    var logs = loi.Select(x => Math.Log(x)).ToArray();
    double moy = logs.Average();
    double variance = logs.Average(x => (x - moy) * (x - moy));
    if (Math.Abs(moy - 0.7) > 0.05 || Math.Abs(variance - 0.64) > 0.07)
        throw new Exception("Moyenne ou variance des logarithmes incorrecte");
    var r = new Random(25);
    int succes = 0;
    for (int j = 0; j < 3000; j++)
    {
        var tirage = ValeursLogNormales(100, 0, 1, r);
        if (tirage == null || tirage.Length != 100 || tirage.Any(x => !double.IsFinite(x) || x <= 0))
            throw new Exception("Tirage manquant, non fini ou non positif");
        var decision = SecretaireValeurs(tirage);
        if (decision == null)
            throw new Exception("Décision manquante");
        if (decision.Value)
            succes++;
    }

    double taux = succes / 3000.0;
    if (taux < 0.32 || taux > 0.42)
        throw new Exception("Succès incompatible avec le seuil de rang");
    Console.WriteLine($"Log-valeurs : moyenne {moy:F3}, variance {variance:F3}; succès {taux:F3} sur 3000 essais");
}


Exercice 3 à compléter


## Conclusion — deux manières de décider sans connaître la suite

| Étape | Expérience exécutée | Portée |
|---|---|---|
| Secrétaire classique | 20 000 permutations pour chacune des trois tailles ; récapitulatif à 5 000 | Fréquence finie comparée à la limite $1/e$ |
| Secrétaire matroïdal | Deux oracles, deux stratégies à 2 000 essais, huit graines sur l'uniforme | Dilemme mesuré pour des instances et poids fixés ; aucune garantie universelle déduite |
| k-server | WFA et optimum par DP exacte, glouton, cycle à trois longueurs et ligne aléatoire | Écart visible sur un adversaire de l'heuristique ; pas de preuve de la borne générale |
| Épistémologie | Annonces et relevé Mathlib rapportés par le Python | Preuves et lecture directe des prépublications hors de ce port |

## À retenir

1. **La baseline utilise l'ordre relatif.** Les trois fréquences affichées concernent $n=20,100,1000$ ; la référence $1/e$ est une limite et non la valeur exacte à chaque petite taille.
2. **L'oracle donne le sens de « choix admissible ».** Dans l'uniforme il limite le nombre d'éléments ; dans le graphique il impose l'absence de cycle. L'optimum hors ligne trie les poids et ne viole jamais cet oracle.
3. **L'échantillonnage a une portée précise.** Huit graines sur les mêmes poids montrent la dispersion des ordres, pas la variabilité de toutes les instances. Les deux heuristiques ne sont pas l'algorithme universel rapporté par le source.
4. **WFA réutilise la fonction de travail du préfixe.** Sur le cycle alterné, les coûts et ratios sont imprimés pour trois longueurs ; sur la ligne aléatoire, ils décrivent un autre régime. Une garantie $c$-compétitive peut inclure une constante additive.

## Vérifiez votre compréhension

1. Pourquoi une fréquence à $n=20$ différente de $1/e$ ne contredit-elle pas la baseline classique ?
2. Pourquoi la stratégie à seuil peut-elle être utile sur l'uniforme mais coûteuse sur la grille graphique étudiée ?
3. Quelles informations la fonction de travail contient-elle, et que signifie une garantie pire cas pour WFA ?

<details><summary>Éléments de réponse</summary>

1. $1/e$ est une limite asymptotique ; à petite taille les corrections finies et le bruit d'échantillonnage restent visibles.
2. L'attente préserve du rang pour de grandes valeurs, mais peut écarter des arêtes utiles ; les contraintes et les poids changent le compromis. Les essais ne prouvent aucune dominance générale.
3. $W_t(S)$ résume le meilleur coût passé pour finir en $S$. La garantie porte sur toute séquence, éventuellement avec une constante additive, et ne découle pas de nos quelques ratios.
</details>

## Pour aller plus loin

[Source Python](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb) · [04 — les baselines](Complexity-04-OnlineAlgorithms-Python.ipynb) · [04c — k-server et WFA](Complexity-04c-KServer-WorkFunction-Python.ipynb) · [04d — secrétaire matroïdal](Complexity-04d-Secretaire-Matroidal-Python.ipynb) · [03b — hiérarchie](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb).

**Références rapportées par le Python** : Singla, *The Matroid Secretary Conjecture is True* ; Coester, Koutsoupias et Zbysiński, *The k-server conjecture is true* ; Babaioff, Immorlica et Kleinberg sur la conjecture de secrétaire matroïdal ; Koutsoupias et Papadimitriou sur WFA $(2k-1)$-compétitif. Ces publications n'ont pas été consultées pour ce port ; les sorties ci-dessus proviennent des calculs .NET.